
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 5L - Working with Complex Data Types in E-Commerce Data

In this lab, you'll practice working with complex data types in Spark, including handling JSON strings, converting them to structured types, and manipulating nested data structures.

## Scenario

You are a data engineer at an e-commerce company that collects data about customer orders, product reviews, and customer browsing behavior. The data contains nested structures that need to be properly processed for analysis.

### Objectives
- Convert JSON string data to Spark SQL native complex types
- Work with arrays and structs
- Use functions like explode, collect_list, and pivot
- Extract and analyze valuable insights from nested data

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.


## A. Classroom Setup

Run the following cell to configure your working environment for this course. It will set your default catalog to **dbacademy** and the schema to your specific schema name shown below using the `USE` statements.

Also, It will create a temp table for you named `ecommerce_raw`
<br></br>

```
USE CATALOG dbacademy;
USE SCHEMA dbacademy.<your unique schema name>;
```

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.

In [0]:
%run ./Includes/Classroom-Setup-5L

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Course Catalog:,
Your Schema:,


#### Querying the newly created table

In [0]:
%sql
select * from ecommerce_raw

customer_id,name,email,registration_date,tags,recent_orders,browsing_history
1001,Jordan Smith,jordan.smith@email.com,2022-03-15,"[""loyal"", ""premium"", ""tech-enthusiast""]","[ {""order_id"": ""O8823"", ""date"": ""2023-01-05"", ""total"": 799.99, ""items"": [ {""product_id"": ""PHONE-256"", ""name"": ""Smartphone XS"", ""price"": 699.99, ""quantity"": 1}, {""product_id"": ""CASE-101"", ""name"": ""Phone Case"", ""price"": 29.99, ""quantity"": 1}, {""product_id"": ""CHGR-201"", ""name"": ""Fast Charger"", ""price"": 49.99, ""quantity"": 1} ]}, {""order_id"": ""O9012"", ""date"": ""2023-02-18"", ""total"": 129.95, ""items"": [ {""product_id"": ""HDPHN-110"", ""name"": ""Wireless Headphones"", ""price"": 129.95, ""quantity"": 1} ]} ]","[""smartphones"", ""accessories"", ""audio"", ""wearables""]"
1002,Alex Johnson,alex.j@email.com,2021-11-20,"[""new"", ""standard"", ""home-office""]","[ {""order_id"": ""O8901"", ""date"": ""2023-01-10"", ""total"": 1299.99, ""items"": [ {""product_id"": ""LAPTOP-15"", ""name"": ""Ultrabook Pro"", ""price"": 1199.99, ""quantity"": 1}, {""product_id"": ""MOUSE-202"", ""name"": ""Ergonomic Mouse"", ""price"": 49.99, ""quantity"": 1}, {""product_id"": ""KYBRD-303"", ""name"": ""Mechanical Keyboard"", ""price"": 89.99, ""quantity"": 1} ]} ]","[""laptops"", ""office-equipment"", ""monitors"", ""storage""]"
1003,Taylor Williams,t.williams@email.com,2022-08-05,"[""standard"", ""gamer""]","[ {""order_id"": ""O9188"", ""date"": ""2023-02-01"", ""total"": 2099.97, ""items"": [ {""product_id"": ""GPU-3080"", ""name"": ""Graphics Card RTX"", ""price"": 899.99, ""quantity"": 1}, {""product_id"": ""CPU-i9"", ""name"": ""Processor i9"", ""price"": 499.99, ""quantity"": 1}, {""product_id"": ""RAM-32GB"", ""name"": ""Gaming RAM 32GB"", ""price"": 189.99, ""quantity"": 2}, {""product_id"": ""MBOARD-Z"", ""name"": ""Gaming Motherboard"", ""price"": 319.99, ""quantity"": 1} ]} ]","[""gaming"", ""pc-components"", ""monitors"", ""accessories""]"
1004,Morgan Lee,morgan.lee@email.com,2022-06-10,"[""standard"", ""photography""]","[ {""order_id"": ""O9021"", ""date"": ""2023-01-15"", ""total"": 3299.98, ""items"": [ {""product_id"": ""CAM-DSLR"", ""name"": ""Professional Camera"", ""price"": 2499.99, ""quantity"": 1}, {""product_id"": ""LENS-50mm"", ""name"": ""Prime Lens"", ""price"": 349.99, ""quantity"": 1}, {""product_id"": ""TRIPOD-P"", ""name"": ""Premium Tripod"", ""price"": 149.99, ""quantity"": 1}, {""product_id"": ""SDCARD-128"", ""name"": ""Memory Card 128GB"", ""price"": 79.99, ""quantity"": 3} ]}, {""order_id"": ""O9254"", ""date"": ""2023-02-28"", ""total"": 299.98, ""items"": [ {""product_id"": ""BAG-CAM"", ""name"": ""Camera Bag"", ""price"": 189.99, ""quantity"": 1}, {""product_id"": ""CLEAN-KIT"", ""name"": ""Lens Cleaning Kit"", ""price"": 29.99, ""quantity"": 1} ]} ]","[""cameras"", ""photography"", ""lenses"", ""accessories""]"
1005,Casey Rivera,casey.r@email.com,2021-09-30,"[""premium"", ""smart-home""]","[ {""order_id"": ""O8765"", ""date"": ""2023-01-02"", ""total"": 1029.95, ""items"": [ {""product_id"": ""SMHUB-01"", ""name"": ""Smart Home Hub"", ""price"": 249.99, ""quantity"": 1}, {""product_id"": ""SMSPK-02"", ""name"": ""Smart Speaker"", ""price"": 179.99, ""quantity"": 2}, {""product_id"": ""SMBLB-03"", ""name"": ""Smart Bulbs Pack"", ""price"": 119.99, ""quantity"": 3}, {""product_id"": ""SMSENS-04"", ""name"": ""Motion Sensors"", ""price"": 89.99, ""quantity"": 1} ]}, {""order_id"": ""O9181"", ""date"": ""2023-02-15"", ""total"": 349.98, ""items"": [ {""product_id"": ""SMDLOCK-05"", ""name"": ""Smart Door Lock"", ""price"": 249.99, ""quantity"": 1}, {""product_id"": ""SMCAM-06"", ""name"": ""Indoor Camera"", ""price"": 99.99, ""quantity"": 1} ]} ]","[""smart-home"", ""security"", ""automation"", ""speakers""]"


## B. Load and Inspect Raw Data with JSON Strings

Load and examine the retail dataset which includes JSON strings.

In [0]:
## Read the sample dataset
events_df = spark.read.table("ecommerce_raw")

## Examine the schema and display sample data
events_df.printSchema()
display(events_df)

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- recent_orders: string (nullable = true)
 |-- browsing_history: string (nullable = true)



customer_id,name,email,registration_date,tags,recent_orders,browsing_history
1001,Jordan Smith,jordan.smith@email.com,2022-03-15,"[""loyal"", ""premium"", ""tech-enthusiast""]","[ {""order_id"": ""O8823"", ""date"": ""2023-01-05"", ""total"": 799.99, ""items"": [ {""product_id"": ""PHONE-256"", ""name"": ""Smartphone XS"", ""price"": 699.99, ""quantity"": 1}, {""product_id"": ""CASE-101"", ""name"": ""Phone Case"", ""price"": 29.99, ""quantity"": 1}, {""product_id"": ""CHGR-201"", ""name"": ""Fast Charger"", ""price"": 49.99, ""quantity"": 1} ]}, {""order_id"": ""O9012"", ""date"": ""2023-02-18"", ""total"": 129.95, ""items"": [ {""product_id"": ""HDPHN-110"", ""name"": ""Wireless Headphones"", ""price"": 129.95, ""quantity"": 1} ]} ]","[""smartphones"", ""accessories"", ""audio"", ""wearables""]"
1002,Alex Johnson,alex.j@email.com,2021-11-20,"[""new"", ""standard"", ""home-office""]","[ {""order_id"": ""O8901"", ""date"": ""2023-01-10"", ""total"": 1299.99, ""items"": [ {""product_id"": ""LAPTOP-15"", ""name"": ""Ultrabook Pro"", ""price"": 1199.99, ""quantity"": 1}, {""product_id"": ""MOUSE-202"", ""name"": ""Ergonomic Mouse"", ""price"": 49.99, ""quantity"": 1}, {""product_id"": ""KYBRD-303"", ""name"": ""Mechanical Keyboard"", ""price"": 89.99, ""quantity"": 1} ]} ]","[""laptops"", ""office-equipment"", ""monitors"", ""storage""]"
1003,Taylor Williams,t.williams@email.com,2022-08-05,"[""standard"", ""gamer""]","[ {""order_id"": ""O9188"", ""date"": ""2023-02-01"", ""total"": 2099.97, ""items"": [ {""product_id"": ""GPU-3080"", ""name"": ""Graphics Card RTX"", ""price"": 899.99, ""quantity"": 1}, {""product_id"": ""CPU-i9"", ""name"": ""Processor i9"", ""price"": 499.99, ""quantity"": 1}, {""product_id"": ""RAM-32GB"", ""name"": ""Gaming RAM 32GB"", ""price"": 189.99, ""quantity"": 2}, {""product_id"": ""MBOARD-Z"", ""name"": ""Gaming Motherboard"", ""price"": 319.99, ""quantity"": 1} ]} ]","[""gaming"", ""pc-components"", ""monitors"", ""accessories""]"
1004,Morgan Lee,morgan.lee@email.com,2022-06-10,"[""standard"", ""photography""]","[ {""order_id"": ""O9021"", ""date"": ""2023-01-15"", ""total"": 3299.98, ""items"": [ {""product_id"": ""CAM-DSLR"", ""name"": ""Professional Camera"", ""price"": 2499.99, ""quantity"": 1}, {""product_id"": ""LENS-50mm"", ""name"": ""Prime Lens"", ""price"": 349.99, ""quantity"": 1}, {""product_id"": ""TRIPOD-P"", ""name"": ""Premium Tripod"", ""price"": 149.99, ""quantity"": 1}, {""product_id"": ""SDCARD-128"", ""name"": ""Memory Card 128GB"", ""price"": 79.99, ""quantity"": 3} ]}, {""order_id"": ""O9254"", ""date"": ""2023-02-28"", ""total"": 299.98, ""items"": [ {""product_id"": ""BAG-CAM"", ""name"": ""Camera Bag"", ""price"": 189.99, ""quantity"": 1}, {""product_id"": ""CLEAN-KIT"", ""name"": ""Lens Cleaning Kit"", ""price"": 29.99, ""quantity"": 1} ]} ]","[""cameras"", ""photography"", ""lenses"", ""accessories""]"
1005,Casey Rivera,casey.r@email.com,2021-09-30,"[""premium"", ""smart-home""]","[ {""order_id"": ""O8765"", ""date"": ""2023-01-02"", ""total"": 1029.95, ""items"": [ {""product_id"": ""SMHUB-01"", ""name"": ""Smart Home Hub"", ""price"": 249.99, ""quantity"": 1}, {""product_id"": ""SMSPK-02"", ""name"": ""Smart Speaker"", ""price"": 179.99, ""quantity"": 2}, {""product_id"": ""SMBLB-03"", ""name"": ""Smart Bulbs Pack"", ""price"": 119.99, ""quantity"": 3}, {""product_id"": ""SMSENS-04"", ""name"": ""Motion Sensors"", ""price"": 89.99, ""quantity"": 1} ]}, {""order_id"": ""O9181"", ""date"": ""2023-02-15"", ""total"": 349.98, ""items"": [ {""product_id"": ""SMDLOCK-05"", ""name"": ""Smart Door Lock"", ""price"": 249.99, ""quantity"": 1}, {""product_id"": ""SMCAM-06"", ""name"": ""Indoor Camera"", ""price"": 99.99, ""quantity"": 1} ]} ]","[""smart-home"", ""security"", ""automation"", ""speakers""]"


## C. Convert JSON Strings to Structured Types

The `tags`, `recent_orders`, and `browsing_history` columns contain JSON strings. Let's convert them to proper Spark structured types.

In [0]:
# 1. Get a sample of the JSON strings in each column
# 2. Infer schemas from the JSON samples
# 3. Convert the JSON strings to structured types using from_json and display the resulting DataFrame

In [0]:
## Get sample JSON strings
tags_json = ecommerce_df.select("tags").limit(1).collect()[0][0]
recent_orders_json = ecommerce_df.select("recent_orders").limit(1).collect()[0][0]
browsing_history_json = ecommerce_df.select("browsing_history").limit(1).collect()[0][0]

print("Tags sample:", tags_json)
print("\nRecent orders sample:", recent_orders_json)
print("\nBrowsing history sample:", browsing_history_json)

Tags sample: ["loyal", "premium", "tech-enthusiast"]

Recent orders sample: [
         {"order_id": "O8823", "date": "2023-01-05", "total": 799.99, "items": [
           {"product_id": "PHONE-256", "name": "Smartphone XS", "price": 699.99, "quantity": 1},
           {"product_id": "CASE-101", "name": "Phone Case", "price": 29.99, "quantity": 1},
           {"product_id": "CHGR-201", "name": "Fast Charger", "price": 49.99, "quantity": 1}
         ]},
         {"order_id": "O9012", "date": "2023-02-18", "total": 129.95, "items": [
           {"product_id": "HDPHN-110", "name": "Wireless Headphones", "price": 129.95, "quantity": 1}
         ]}
       ]

Browsing history sample: ["smartphones", "accessories", "audio", "wearables"]


In [0]:
## Infer schemas from the JSON samples

## Define/infer schemas
tags_schema = schema_of_json(lit(tags_json))
recent_orders_schema = schema_of_json(lit(recent_orders_json))
browsing_history_schema = schema_of_json(lit(browsing_history_json))

In [0]:
parsed_df = ecommerce_df.select(
    "customer_id",
    "name",
    "email",
    "registration_date",
    from_json("tags", tags_schema).alias("tags"),
    from_json("recent_orders", recent_orders_schema).alias("recent_orders"),
    from_json("browsing_history", browsing_history_schema).alias("browsing_history")
)

## Examine the schema and display sample data
parsed_df.printSchema()
display(parsed_df)

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- recent_orders: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- items: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |    |-- price: double (nullable = true)
 |    |    |    |    |-- product_id: string (nullable = true)
 |    |    |    |    |-- quantity: long (nullable = true)
 |    |    |-- order_id: string (nullable = true)
 |    |    |-- total: double (nullable = true)
 |-- browsing_history: array (nullable = true)
 |    |-- element: string (containsNull = true)



customer_id,name,email,registration_date,tags,recent_orders,browsing_history
1001,Jordan Smith,jordan.smith@email.com,2022-03-15,"List(loyal, premium, tech-enthusiast)","List(List(2023-01-05, List(List(Smartphone XS, 699.99, PHONE-256, 1), List(Phone Case, 29.99, CASE-101, 1), List(Fast Charger, 49.99, CHGR-201, 1)), O8823, 799.99), List(2023-02-18, List(List(Wireless Headphones, 129.95, HDPHN-110, 1)), O9012, 129.95))","List(smartphones, accessories, audio, wearables)"
1002,Alex Johnson,alex.j@email.com,2021-11-20,"List(new, standard, home-office)","List(List(2023-01-10, List(List(Ultrabook Pro, 1199.99, LAPTOP-15, 1), List(Ergonomic Mouse, 49.99, MOUSE-202, 1), List(Mechanical Keyboard, 89.99, KYBRD-303, 1)), O8901, 1299.99))","List(laptops, office-equipment, monitors, storage)"
1003,Taylor Williams,t.williams@email.com,2022-08-05,"List(standard, gamer)","List(List(2023-02-01, List(List(Graphics Card RTX, 899.99, GPU-3080, 1), List(Processor i9, 499.99, CPU-i9, 1), List(Gaming RAM 32GB, 189.99, RAM-32GB, 2), List(Gaming Motherboard, 319.99, MBOARD-Z, 1)), O9188, 2099.97))","List(gaming, pc-components, monitors, accessories)"
1004,Morgan Lee,morgan.lee@email.com,2022-06-10,"List(standard, photography)","List(List(2023-01-15, List(List(Professional Camera, 2499.99, CAM-DSLR, 1), List(Prime Lens, 349.99, LENS-50mm, 1), List(Premium Tripod, 149.99, TRIPOD-P, 1), List(Memory Card 128GB, 79.99, SDCARD-128, 3)), O9021, 3299.98), List(2023-02-28, List(List(Camera Bag, 189.99, BAG-CAM, 1), List(Lens Cleaning Kit, 29.99, CLEAN-KIT, 1)), O9254, 299.98))","List(cameras, photography, lenses, accessories)"
1005,Casey Rivera,casey.r@email.com,2021-09-30,"List(premium, smart-home)","List(List(2023-01-02, List(List(Smart Home Hub, 249.99, SMHUB-01, 1), List(Smart Speaker, 179.99, SMSPK-02, 2), List(Smart Bulbs Pack, 119.99, SMBLB-03, 3), List(Motion Sensors, 89.99, SMSENS-04, 1)), O8765, 1029.95), List(2023-02-15, List(List(Smart Door Lock, 249.99, SMDLOCK-05, 1), List(Indoor Camera, 99.99, SMCAM-06, 1)), O9181, 349.98))","List(smart-home, security, automation, speakers)"


## D. Working with Arrays

Now that we have proper structured data, let's analyze the customer tags and browsing history.

In [0]:
# 1. Calculate the number of tags and browsing history items for each customer
# 2. Explode the tags array to see all unique customer tags
# 3. Find the most common browsing categories across all customers
# HINT: use the `array_size` function or its alias `size`

In [0]:
## Calculate the number of tags and browsing history items for each customer
array_sizes_df = parsed_df.select(
    "customer_id",
    "name",
    size("tags").alias("num_tags"),
    size("browsing_history").alias("num_browsing_categories")
)

display(array_sizes_df)

customer_id,name,num_tags,num_browsing_categories
1001,Jordan Smith,3,4
1002,Alex Johnson,3,4
1003,Taylor Williams,2,4
1004,Morgan Lee,2,4
1005,Casey Rivera,2,4


In [0]:
## Explode tags to see all customer categorizations
exploded_tags_df = parsed_df.select(
    "customer_id",
    "name",
    explode("tags").alias("tag")
)

display(exploded_tags_df)

customer_id,name,tag
1001,Jordan Smith,loyal
1001,Jordan Smith,premium
1001,Jordan Smith,tech-enthusiast
1002,Alex Johnson,new
1002,Alex Johnson,standard
1002,Alex Johnson,home-office
1003,Taylor Williams,standard
1003,Taylor Williams,gamer
1004,Morgan Lee,standard
1004,Morgan Lee,photography


In [0]:
## Find the most common customer tags
## Count frequency of each tag
tag_counts_df = exploded_tags_df.groupBy("tag").count().orderBy(desc("count"))
display(tag_counts_df)

tag,count
standard,3
premium,2
tech-enthusiast,1
loyal,1
new,1
home-office,1
gamer,1
smart-home,1
photography,1


In [0]:
# 1. Explode the recent_orders array to analyze individual orders
# 2. Calculate total revenue per customer

In [0]:
## Explode recent_orders to analyze individual orders
orders_df = parsed_df.select(
    "customer_id",
    "name",
    explode("recent_orders").alias("order")
)

## Calculate total revenue per customer
customer_revenue_df = orders_df.groupBy(
    "customer_id",
    "name"
).agg(
    sum("order.total").alias("total_revenue"),
    count("order.order_id").alias("order_count")
).orderBy(desc("total_revenue"))

display(customer_revenue_df)

customer_id,name,total_revenue,order_count
1004,Morgan Lee,3599.96,2
1003,Taylor Williams,2099.97,1
1005,Casey Rivera,1379.93,2
1002,Alex Johnson,1299.99,1
1001,Jordan Smith,929.94,2


## E. Bonus Challenge: Analyze Customer Purchasing Patterns

Let's use the `collect_list` and `collect_set` aggregate functions to create summaries of customer purchasing patterns.

In [0]:
## First, create a flattened view of orders
order_items_df = orders_df.select(
    "customer_id",
    "name",
    "order.order_id",
    "order.date",
    explode("order.items").alias("item")
)

## Now extract the name field from each item
item_details_df = order_items_df.selectExpr(
    "customer_id",
    "name",
    "item.name as product_name"
)

# Inspect the data
display(item_details_df)

customer_id,name,product_name
1001,Jordan Smith,Smartphone XS
1001,Jordan Smith,Phone Case
1001,Jordan Smith,Fast Charger
1001,Jordan Smith,Wireless Headphones
1002,Alex Johnson,Ultrabook Pro
1002,Alex Johnson,Ergonomic Mouse
1002,Alex Johnson,Mechanical Keyboard
1003,Taylor Williams,Graphics Card RTX
1003,Taylor Williams,Processor i9
1003,Taylor Williams,Gaming RAM 32GB


In [0]:
## Collect all products purchased by each customer, creating new columns called "all_products_purchased" and "unique_products_purchased" for each "customer_id"
customer_products_df = item_details_df.groupBy(
    "customer_id"
).agg(
    collect_list("product_name").alias("all_products_purchased"),
    collect_set("product_name").alias("unique_products_purchased")
)

display(customer_products_df)

customer_id,all_products_purchased,unique_products_purchased
1001,"List(Smartphone XS, Phone Case, Fast Charger, Wireless Headphones)","List(Wireless Headphones, Fast Charger, Smartphone XS, Phone Case)"
1002,"List(Ultrabook Pro, Ergonomic Mouse, Mechanical Keyboard)","List(Ultrabook Pro, Mechanical Keyboard, Ergonomic Mouse)"
1003,"List(Graphics Card RTX, Processor i9, Gaming RAM 32GB, Gaming Motherboard)","List(Processor i9, Gaming Motherboard, Gaming RAM 32GB, Graphics Card RTX)"
1004,"List(Professional Camera, Prime Lens, Premium Tripod, Memory Card 128GB, Camera Bag, Lens Cleaning Kit)","List(Camera Bag, Prime Lens, Professional Camera, Premium Tripod, Lens Cleaning Kit, Memory Card 128GB)"
1005,"List(Smart Home Hub, Smart Speaker, Smart Bulbs Pack, Motion Sensors, Smart Door Lock, Indoor Camera)","List(Smart Home Hub, Smart Speaker, Smart Door Lock, Motion Sensors, Indoor Camera, Smart Bulbs Pack)"



&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
